# Techincal Requirements
## Part A: Data Preprocessing Pipeline

Covers all 4 spec requirements:
1.  Text cleaning (imported from EDA)
2.  Tokenisation & vocabulary construction
3.  Encoding & padding
4.  Dataset splitting (70/10/20 stratified)

## 0. Imports and Constants

In [4]:
import re
import torch
from collections import Counter
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Special token indices
PAD, UNK = 0, 1

# ── Hyperparameters to experiment with (spec says N ∈ {10000, 20000, 30000}) ──
MAX_VOCAB_SIZE = 44_762   # try 10_000 / 20_000 / 30_000
MAX_LEN        = 454      # from EDA p90; try 128 / 256 / 512
BATCH_SIZE     = 64
SEED           = 42

print(f'Config: vocab={MAX_VOCAB_SIZE:,}  max_len={MAX_LEN}  batch={BATCH_SIZE}')

Config: vocab=44,762  max_len=454  batch=64


## 1. Text Cleaning (Reused from 01_eda.ipynb)

In [5]:
def clean_text(text: str) -> str:
    """Remove HTML tags, keep only letters/digits/spaces, lowercase."""
    text = re.sub(r'<[^>]+>', ' ', text)          # strip HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)   # keep letters & digits only
    return text.lower().strip()

# ── Load & clean IMDb ──────────────────────────────────────────────────────────
dataset = load_dataset('imdb')

# Combine train + test so we can do our own 70/10/20 split
all_texts  = ([clean_text(ex['text']) for ex in dataset['train']] +
              [clean_text(ex['text']) for ex in dataset['test']])
all_labels = ([ex['label'] for ex in dataset['train']] +
              [ex['label'] for ex in dataset['test']])

print(f'Total samples after cleaning: {len(all_texts):,}')
print(f'Sample: "{all_texts[0][:120]}..."')

Total samples after cleaning: 50,000
Sample: "i rented i am curious yellow from my video store because of all the controversy that surrounded it when it was first rel..."


## 2. Tokenisation and vocabulary construction 

In [6]:
def build_vocab(texts: list[str], max_size: int = 20_000) -> dict[str, int]:
    """
    Build word→index vocabulary from a list of cleaned texts.
    Reserves index 0 for <pad> and index 1 for <unk>.
    Returns the top (max_size - 2) most frequent tokens.
    """
    counter = Counter(w for text in texts for w in text.split())
    vocab   = {'<pad>': PAD, '<unk>': UNK}
    for word, _ in counter.most_common(max_size - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(all_texts, max_size=MAX_VOCAB_SIZE)

print(f'Vocabulary size : {len(vocab):,}  (including <pad> and <unk>)')
print(f'Top-10 entries  : {list(vocab.items())[2:12]}')

Vocabulary size : 44,762  (including <pad> and <unk>)
Top-10 entries  : [('the', 2), ('and', 3), ('a', 4), ('of', 5), ('to', 6), ('is', 7), ('it', 8), ('in', 9), ('i', 10), ('this', 11)]


## 3. Encoding and padding

In [7]:
class SentimentDataset(Dataset):
    """
    PyTorch Dataset for IMDb sentiment.
    Encodes text → integer IDs and truncates to max_len at construction time.
    Padding to a uniform length within each batch is handled by collate_fn.
    """

    def __init__(self, texts: list[str], labels: list[int],
                 vocab: dict[str, int], max_len: int = 256):
        self.data = []
        for text, label in zip(texts, labels):
            ids = [vocab.get(w, UNK) for w in text.split()]  # encode
            ids = ids[:max_len]                               # truncate if too long
            self.data.append((
                torch.tensor(ids,   dtype=torch.long),
                torch.tensor(label, dtype=torch.long)
            ))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]


def collate_fn(batch):
    """
    Dynamic batch padding: pads all sequences in the batch to the length
    of the longest sequence in that batch (not the global max_len).
    This avoids wasting compute on excessive padding for short-review batches.
    """
    seqs, labels = zip(*batch)
    seqs_padded  = pad_sequence(seqs, batch_first=True, padding_value=PAD)
    return seqs_padded, torch.stack(labels)


# Quick encode sanity check
sample_ids = [vocab.get(w, UNK) for w in all_texts[0].split()][:10]
print('First 10 token IDs of sample[0]:', sample_ids)
print('Decoded back                    :', [list(vocab.keys())[list(vocab.values()).index(i)] for i in sample_ids])

First 10 token IDs of sample[0]: [10, 1568, 10, 236, 2114, 3832, 38, 59, 374, 1073]
Decoded back                    : ['i', 'rented', 'i', 'am', 'curious', 'yellow', 'from', 'my', 'video', 'store']


## 4. Dataset splitting

In [8]:
from sklearn.model_selection import train_test_split

indices = list(range(len(all_texts)))

# Step 1: split off 20% test
idx_trainval, idx_test = train_test_split(
    indices, test_size=0.20, stratify=all_labels, random_state=SEED
)

# Step 2: split remaining 80% into 70% train + 10% val
# 10% of total = 10/80 = 12.5% of trainval
labels_trainval = [all_labels[i] for i in idx_trainval]
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.125, stratify=labels_trainval, random_state=SEED
)

# Verify split sizes
total = len(all_texts)
print(f'Total  : {total:,}')
print(f'Train  : {len(idx_train):,}  ({len(idx_train)/total*100:.1f}%)')
print(f'Val    : {len(idx_val):,}   ({len(idx_val)/total*100:.1f}%)')
print(f'Test   : {len(idx_test):,}  ({len(idx_test)/total*100:.1f}%)')

# Verify class balance is preserved
for name, idx_split in [('Train', idx_train), ('Val', idx_val), ('Test', idx_test)]:
    labels_split = [all_labels[i] for i in idx_split]
    pos_pct = sum(labels_split) / len(labels_split) * 100
    print(f'  {name} positive ratio: {pos_pct:.1f}%')

Total  : 50,000
Train  : 35,000  (70.0%)
Val    : 5,000   (10.0%)
Test   : 10,000  (20.0%)
  Train positive ratio: 50.0%
  Val positive ratio: 50.0%
  Test positive ratio: 50.0%


HELLO, PAI